# MLP Baseline: Train and Common Evaluation trên Google Colab

Notebook này tham khảo cùng cấu hình Colab/Google Drive của `drl_product_recommendation_train_val_test_colab_9_6.ipynb` và chạy riêng MLP baseline.

Tạo cấu trúc sau trên Google Drive trước khi chạy:

```text
MyDrive/drl-product-recommendation/
├── colab_inputs/
│   ├── orders.csv
│   ├── products.csv
│   ├── order_products__prior.csv
│   └── order_products__train.csv
└── colab_outputs/
```

Notebook tự preprocess và split giống notebook DQN, train MLP trên train, chọn checkpoint bằng validation, final test trên common deterministic windows, rồi lưu kết quả vào `colab_outputs/mlp_baseline/` và file zip vào `colab_outputs/`.

Cấu hình Colab mặc định lấy mẫu cố định 2 triệu training windows và 200 nghìn validation/test windows để hoàn thành trong thời gian hợp lý. Không nên bật `FULL_DATA_TRAIN=True` trừ khi bạn chấp nhận mỗi epoch có thể chạy nhiều giờ.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if 'display' not in globals():
    from IPython.display import display

REPO_URL = 'https://github.com/dangnguyenhoai/drl-product-recommendation.git'
BRANCH = 'Quynh'

def run_command(command, cwd=None):
    print('\n>>>', ' '.join(map(str, command)))
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with return code {result.returncode}: {command}')
    return result

if IN_COLAB:
    print('Môi trường: Google Colab')
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_ROOT = Path('/content/drl-product-recommendation')
    RAW_DIR = Path('/content/drive/MyDrive/drl-product-recommendation/colab_inputs')
    DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/drl-product-recommendation/colab_outputs')

    os.chdir('/content')
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    run_command(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_ROOT)], cwd='/content')
    os.chdir(PROJECT_ROOT)
    run_command([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'matplotlib', 'ipython', 'tabulate'])
else:
    print('Môi trường: Local')
    PROJECT_ROOT = next(
        (path for path in [Path.cwd(), *Path.cwd().parents]
         if (path / 'baseline').is_dir() and (path / 'evaluation').is_dir()),
        None,
    )
    if PROJECT_ROOT is None:
        raise RuntimeError('Không tìm thấy project root.')
    RAW_DIR = PROJECT_ROOT / 'data'
    DRIVE_OUTPUT_DIR = PROJECT_ROOT / 'outputs'

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
required_project_files = [
    Path('baseline/baseline_train_v3.py'),
    Path('evaluation/evaluate_mlp_common.py'),
    Path('evaluation/evaluate_models_common.py'),
    Path('utils/split_history_train_val_test.py'),
]
for path in required_project_files:
    if not path.is_file():
        raise FileNotFoundError(f'Clone sai branch hoặc thiếu file: {path}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_DIR:', RAW_DIR)
print('DRIVE_OUTPUT_DIR:', DRIVE_OUTPUT_DIR)
print('Python:', sys.executable)
print('DEVICE:', DEVICE)
run_command([sys.executable, '--version'])
if DEVICE == 'cuda':
    run_command(['nvidia-smi'])

## 1. Cấu hình

`ACTION_DIM` sẽ được suy ra từ cả ba split để khớp global action space. Với dữ liệu hiện tại giá trị mong đợi là `1000`.

In [ ]:
QUICK_RUN = not IN_COLAB
FULL_DATA_TRAIN = False  # True sẽ dùng toàn bộ ~9.7M mẫu/epoch và có thể mất nhiều giờ mỗi epoch.
SEED = 42

STATE_SIZE = 5
TOP_K = 5
TOP_N_ITEMS = 1000
EMBEDDING_DIM = 32
HIDDEN_DIMS = '128'
BATCH_SIZE = 1024 if IN_COLAB else 64
LEARNING_RATE = 1e-4
EPOCHS = 10
PATIENCE = 3
NUM_WORKERS = 2 if IN_COLAB else 0
LOG_EVERY_BATCHES = 250

TRAIN_PATH = Path('data/processed/train_history.pkl')
VAL_PATH = Path('data/processed/val_history.pkl')
TEST_PATH = Path('data/processed/test_history.pkl')
MLP_MODEL_NAME = 'mlp_common_baseline.pth'
MLP_MODEL_PATH = Path('outputs/checkpoints') / MLP_MODEL_NAME

# Trên Colab, cell so sánh sẽ lấy model DQN đã chọn bởi notebook DQN từ Drive.
DQN_MODEL_PATH = Path('outputs/checkpoints/dqn_selected_from_drive.pth') if IN_COLAB else Path('outputs/checkpoints/dqn_pure_stable_best.pth')
DQN_RECENT_BOOST = 0.0
RUN_DQN_COMPARISON = not QUICK_RUN

RUN_EPOCHS = 2 if QUICK_RUN else EPOCHS
MAX_TRAIN_SAMPLES = 100_000 if QUICK_RUN else (None if FULL_DATA_TRAIN else 2_000_000)
MAX_EVAL_SAMPLES = 10_000 if QUICK_RUN else (None if FULL_DATA_TRAIN else 200_000)
MAX_COMMON_WINDOWS = 10_000 if QUICK_RUN else None

## 2. Preprocess, split dữ liệu và kiểm tra global action space

Nếu `data/processed/train_history.pkl`, `val_history.pkl`, `test_history.pkl` chưa tồn tại trong repo Colab mới clone, cell này đọc 4 CSV từ `colab_inputs`, giữ Top-1000 item và split temporal `70% / 15% / 15%` giống notebook DQN.

In [ ]:
import gc
import pickle

TRAIN_PATH = Path('data/processed/train_history.pkl')
VAL_PATH = Path('data/processed/val_history.pkl')
TEST_PATH = Path('data/processed/test_history.pkl')
INDEXED_PATH = Path('data/processed/indexed_history.pkl')
split_paths = {'train': TRAIN_PATH, 'validation': VAL_PATH, 'test': TEST_PATH}

if not all(path.is_file() for path in split_paths.values()):
    required_raw_files = {
        'orders': RAW_DIR / 'orders.csv',
        'products': RAW_DIR / 'products.csv',
        'prior': RAW_DIR / 'order_products__prior.csv',
        'train': RAW_DIR / 'order_products__train.csv',
    }
    for name, path in required_raw_files.items():
        if not path.is_file():
            raise FileNotFoundError(f'Không thấy file {name}: {path}')

    print('Đọc raw CSV từ Drive...')
    orders = pd.read_csv(required_raw_files['orders'])
    products = pd.read_csv(required_raw_files['products'])
    prior = pd.read_csv(required_raw_files['prior'])
    train_orders = pd.read_csv(required_raw_files['train'])
    print('orders:', orders.shape)
    print('products:', products.shape)
    print('prior:', prior.shape)
    print('train:', train_orders.shape)

    order_products = pd.concat([prior, train_orders], ignore_index=True)
    merged = order_products.merge(
        orders[['order_id', 'user_id', 'order_number']],
        on='order_id',
        how='inner',
    )
    sort_cols = ['user_id', 'order_number']
    if 'add_to_cart_order' in merged.columns:
        sort_cols.append('add_to_cart_order')
    merged = merged.sort_values(sort_cols)

    top_items = merged['product_id'].value_counts().head(TOP_N_ITEMS).index.tolist()
    item_to_index = {product_id: idx for idx, product_id in enumerate(top_items)}
    top_item_set = set(top_items)
    filtered = merged[merged['product_id'].isin(top_item_set)].copy()
    user_history_raw = filtered.groupby('user_id')['product_id'].apply(list).to_dict()
    indexed_history = {
        user_id: indexed_items
        for user_id, items in user_history_raw.items()
        if len(indexed_items := [item_to_index[item] for item in items if item in item_to_index]) >= 5
    }

    INDEXED_PATH.parent.mkdir(parents=True, exist_ok=True)
    with INDEXED_PATH.open('wb') as file:
        pickle.dump(indexed_history, file)
    print('Saved:', INDEXED_PATH)
    print('Users after filtering:', len(indexed_history))
    print('Interactions:', sum(len(history) for history in indexed_history.values()))

    del orders, products, prior, train_orders, order_products, merged, filtered, user_history_raw, indexed_history
    gc.collect()

    run_command([
        sys.executable, '-m', 'utils.split_history_train_val_test',
        '--input_path', str(INDEXED_PATH),
        '--train_output_path', str(TRAIN_PATH),
        '--val_output_path', str(VAL_PATH),
        '--test_output_path', str(TEST_PATH),
        '--train_ratio', '0.7',
        '--val_ratio', '0.15',
        '--test_ratio', '0.15',
        '--min_history_len', '11',
    ])
else:
    print('Đã có train/validation/test split. Bỏ qua preprocess.')

from baseline.baseline_train_v3 import load_history_pickle

summary_rows = []
global_max_item = -1

for split, path in split_paths.items():
    histories = load_history_pickle(path)
    interactions = sum(len(history) for history in histories)
    max_item = max(int(history.max()) for history in histories)
    unique_items = len({int(item) for history in histories for item in history})
    global_max_item = max(global_max_item, max_item)
    summary_rows.append({
        'split': split,
        'histories': len(histories),
        'interactions': interactions,
        'unique_items': unique_items,
        'max_item_id': max_item,
    })
    del histories

ACTION_DIM = global_max_item + 1
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print('GLOBAL ACTION_DIM:', ACTION_DIM)

## 3. Train MLP baseline

MLP nhận embedding của 5 item gần nhất, flatten, đi qua hidden layer 128 và dự đoán item tiếp theo bằng cross-entropy. Script lưu checkpoint tốt nhất theo validation HitRate@5, training log và learning curves.

In [ ]:
train_command = [
    sys.executable, '-m', 'baseline.baseline_train_v3',
    '--train-data-path', str(TRAIN_PATH),
    '--val-data-path', str(VAL_PATH),
    '--test-data-path', str(TEST_PATH),
    '--model-name', MLP_MODEL_NAME,
    '--state-size', str(STATE_SIZE),
    '--top-k', str(TOP_K),
    '--action-dim', str(ACTION_DIM),
    '--epochs', str(RUN_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--embedding-dim', str(EMBEDDING_DIM),
    '--hidden-dims', HIDDEN_DIMS,
    '--lr', str(LEARNING_RATE),
    '--patience', str(PATIENCE),
    '--num-workers', str(NUM_WORKERS),
    '--log-every-batches', str(LOG_EVERY_BATCHES),
    '--seed', str(SEED),
    '--device', DEVICE,
]
if MAX_TRAIN_SAMPLES is not None:
    train_command += ['--max-train-samples', str(MAX_TRAIN_SAMPLES)]
if MAX_EVAL_SAMPLES is not None:
    train_command += ['--max-eval-samples', str(MAX_EVAL_SAMPLES)]

print(' '.join(train_command))
train_env = os.environ.copy()
train_env['PYTHONUNBUFFERED'] = '1'
subprocess.run(train_command, check=True, env=train_env)

## 4. Xem learning curves và kết quả supervised test

Các metric ở cell này dùng immediate-next-item target. Phần đánh giá chung next-5-window để so sánh công bằng với DQN nằm ở cell kế tiếp.

In [ ]:
history_path = Path('outputs/logs/train_mlp_history_baseline.csv')
supervised_test_path = Path('outputs/logs/mlp_final_test_results.csv')

history_df = pd.read_csv(history_path)
display(history_df.tail())
display(pd.read_csv(supervised_test_path))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train loss')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='validation loss')
axes[0].set_title('MLP loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
metric = f'val_hitrate@{TOP_K}'
axes[1].plot(history_df['epoch'], history_df[metric], label=metric)
axes[1].set_title('Validation HitRate')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Final MLP test trên common deterministic environment

Evaluator này dùng đúng implementation window, valid-action mask, reward và metric từ `evaluation.evaluate_models_common`. Vì vậy kết quả có thể đặt cạnh DQN mà không bị lệch tập test hoặc công thức metric.

In [ ]:
mlp_eval_command = [
    sys.executable, '-m', 'evaluation.evaluate_mlp_common',
    '--data-path', str(TEST_PATH),
    '--model-path', str(MLP_MODEL_PATH),
    '--state-size', str(STATE_SIZE),
    '--top-k', str(TOP_K),
    '--device', DEVICE,
]
if MAX_COMMON_WINDOWS is not None:
    mlp_eval_command += ['--max-windows', str(MAX_COMMON_WINDOWS)]

print(' '.join(mlp_eval_command))
subprocess.run(mlp_eval_command, check=True)
display(pd.read_csv('outputs/logs/mlp_common_test_results.csv'))

## 6. Tùy chọn: so sánh trực tiếp MLP và DQN

Cell này chỉ chạy khi checkpoint DQN tồn tại. Hai model được chấm trên chính xác cùng từng test state và next-5 target.

In [ ]:
import json

if IN_COLAB:
    drive_dqn_dir = DRIVE_OUTPUT_DIR / 'evaluation' / 'latest'
    drive_dqn_model = drive_dqn_dir / 'best_selected_by_validation.pth'
    drive_dqn_metadata = drive_dqn_dir / 'best_selected_by_validation.json'

    if drive_dqn_model.is_file():
        DQN_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_dqn_model, DQN_MODEL_PATH)
        print('Đã copy DQN checkpoint từ Drive:', drive_dqn_model)
    if drive_dqn_metadata.is_file():
        metadata = json.loads(drive_dqn_metadata.read_text(encoding='utf-8'))
        DQN_RECENT_BOOST = float(metadata.get('recent_boost', 0.0))
        print('DQN recent_boost từ metadata:', DQN_RECENT_BOOST)

if RUN_DQN_COMPARISON and DQN_MODEL_PATH.is_file():
    compare_command = [
        sys.executable, '-m', 'evaluation.evaluate_models_common',
        '--data-path', str(TEST_PATH),
        '--mlp-model-path', str(MLP_MODEL_PATH),
        '--dqn-model-path', str(DQN_MODEL_PATH),
        '--state-size', str(STATE_SIZE),
        '--top-k', str(TOP_K),
        '--recent-boost', str(DQN_RECENT_BOOST),
        '--device', DEVICE,
    ]
    print(' '.join(compare_command))
    subprocess.run(compare_command, check=True)
    comparison_df = pd.read_csv('outputs/logs/common_model_comparison.csv')
    display(comparison_df)
elif not RUN_DQN_COMPARISON:
    print('Bỏ qua so sánh DQN trong QUICK_RUN. Đặt RUN_DQN_COMPARISON=True để chạy.')
else:
    print(f'Bỏ qua so sánh DQN vì chưa có checkpoint: {DQN_MODEL_PATH}')

## 7. Đồng bộ checkpoint, metric, biểu đồ và file zip về Google Drive

Giống notebook DQN, cell này lưu một bản `latest`, một bản theo timestamp và zip toàn bộ `outputs/` vào `MyDrive/drl-product-recommendation/colab_outputs/`.

In [ ]:
import json
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_config = {
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'branch': BRANCH,
    'quick_run': QUICK_RUN,
    'device': DEVICE,
    'state_size': STATE_SIZE,
    'top_k': TOP_K,
    'action_dim': ACTION_DIM,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dims': HIDDEN_DIMS,
    'batch_size': BATCH_SIZE,
    'full_data_train': FULL_DATA_TRAIN,
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'max_eval_samples': MAX_EVAL_SAMPLES,
    'num_workers': NUM_WORKERS,
    'learning_rate': LEARNING_RATE,
    'epochs': RUN_EPOCHS,
    'patience': PATIENCE,
    'seed': SEED,
}
config_path = Path('outputs/logs/mlp_run_config.json')
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding='utf-8')

result_files = [
    MLP_MODEL_PATH,
    Path('outputs/logs/train_mlp_history_baseline.csv'),
    Path('outputs/logs/mlp_final_test_results.csv'),
    Path('outputs/logs/mlp_common_test_results.csv'),
    Path('outputs/logs/mlp_common_test_report.md'),
    Path('outputs/logs/common_model_comparison.csv'),
    Path('outputs/logs/common_model_comparison.md'),
    Path('outputs/plots/mlp_loss_curve.png'),
    Path('outputs/plots/mlp_hitrate_at_5_curve.png'),
    Path('outputs/plots/mlp_recall_at_5_curve.png'),
    Path('outputs/plots/mlp_ndcg_at_5_curve.png'),
    config_path,
]
result_files = [path for path in result_files if path.is_file()]

if IN_COLAB:
    drive_mlp_root = DRIVE_OUTPUT_DIR / 'mlp_baseline'
    drive_latest_dir = drive_mlp_root / 'latest'
    drive_timestamp_dir = drive_mlp_root / timestamp

    for target_dir in [drive_latest_dir, drive_timestamp_dir]:
        target_dir.mkdir(parents=True, exist_ok=True)
        for source_path in result_files:
            shutil.copy2(source_path, target_dir / source_path.name)

    zip_name = f'mlp_baseline_outputs_{timestamp}'
    zip_base = Path('/content') / zip_name
    shutil.make_archive(str(zip_base), 'zip', 'outputs')
    drive_zip_path = DRIVE_OUTPUT_DIR / f'{zip_name}.zip'
    shutil.copy2(f'{zip_base}.zip', drive_zip_path)

    print('Saved MLP latest results to:', drive_latest_dir)
    print('Saved timestamped results to:', drive_timestamp_dir)
    print('Saved outputs zip to:', drive_zip_path)
else:
    print('Running locally. Results remain under outputs/.')

print('\nFiles ready for reporting:')
for path in result_files:
    print('-', path)

## Cấu trúc kết quả trên Drive

```text
MyDrive/drl-product-recommendation/colab_outputs/
├── evaluation/latest/                     # DQN notebook lưu model được chọn ở đây
├── mlp_baseline/latest/                   # kết quả MLP mới nhất
├── mlp_baseline/YYYYMMDD_HHMMSS/          # backup từng lần chạy
└── mlp_baseline_outputs_YYYYMMDD_HHMMSS.zip
```

Trong `mlp_baseline/latest/` có checkpoint tốt nhất, training history, common test metrics, report Markdown, biểu đồ và file cấu hình chạy.